# Step 5: Polyphonic Audio Synthesis (MIDI → WAV)

This notebook synthesizes MIDI note frequencies into 44.1kHz WAV piano audio with attack and decay envelopes.


In [ ]:
# Setup environment & imports
import os
import sys
import wave
import math
import struct
import music21

backend_path = r"c:\Users\hamza\Desktop\S2S\backend"
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)
project_root = r"c:\Users\hamza\Desktop\S2S"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

midi_input = "empty_core_output.mid"
if not os.path.exists(midi_input):
    print(f"⚠️ Warning: '{midi_input}' not found. Please run Notebook 04 first.")
else:
    print(f"✅ Input MIDI file found: '{midi_input}'")

In [ ]:
# Synthesize MIDI to 44.1kHz WAV
midi_output_path = "empty_core_output.mid"
wav_output_path = "empty_core_output.wav"

def synthesize_score_to_wav(midi_path, wav_path, sample_rate=44100):
    if not os.path.exists(midi_path):
        raise FileNotFoundError(f"MIDI file '{midi_path}' not found. Run Notebook 04 first.")
        
    score_data = music21.converter.parse(midi_path)
    notes = score_data.flatten().notes
    
    sec_per_beat = 60.0 / 120.0
    events = []
    for el in notes:
        onset = float(el.offset) * sec_per_beat
        dur = max(0.25, float(el.quarterLength) * sec_per_beat)
        if isinstance(el, music21.note.Note):
            events.append((onset, dur, el.pitch.frequency))
        elif isinstance(el, music21.chord.Chord):
            for p in el.pitches:
                events.append((onset, dur, p.frequency))
                
    if not events:
        raise ValueError("No playable MIDI notes found in score.")
        
    total_dur = max(3.0, max(start + d for start, d, _ in events) + 1.0)
    num_samples = int(sample_rate * total_dur)
    buf = [0.0] * num_samples
    
    for onset, dur, freq in events:
        s_idx = int(onset * sample_rate)
        e_idx = min(num_samples, s_idx + int((dur + 0.6) * sample_rate))
        for i in range(s_idx, e_idx):
            t = (i - s_idx) / sample_rate
            env = t / 0.008 if t < 0.008 else math.exp(-2.2 * (t - 0.008) / dur)
            harm = env * 0.12 * (
                math.sin(2 * math.pi * freq * t) +
                0.45 * math.sin(2 * math.pi * 2 * freq * t) +
                0.20 * math.sin(2 * math.pi * 3 * freq * t)
            )
            buf[i] += harm
            
    max_v = max(abs(x) for x in buf) or 1.0
    norm = 0.85 / max_v
    
    with wave.open(wav_path, "w") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        raw = bytearray()
        for s in buf:
            v_int = int(s * norm * 32767)
            raw.extend(struct.pack("<h", max(-32768, min(32767, v_int))))
        wf.writeframes(raw)
        
    print("Successfully synthesized WAV audio:", wav_path, "size:", os.path.getsize(wav_path), "bytes")
    return wav_path

synthesize_score_to_wav(midi_output_path, wav_output_path)